# edge-of-the-world — getting started

`bw_eotw` is an alternate Brightway database backend that lets a single stored edge expand into **multiple matrix cells** at calculation time. Multiple expansions are possible, and some are controlled by a config dict.

This notebook builds a small model — *1 kWh of electricity demand for a widget* — and shows how each interpreter changes the LCA score.

**Model overview**
- `biosphere` — one emission flow: CO₂ (kg)
- `background` — three electricity providers, each with a direct CO₂ emission factor:
  - Coal power: **0.82 kg CO₂/kWh**
  - Wind power: **0.007 kg CO₂/kWh**
  - Solar power: **0.040 kg CO₂/kWh**
- `demo` (eotw backend) — one widget node; its electricity edge is rewritten for each section
- Method GWP: CO₂ characterisation factor = 1 kg CO₂-eq/kg

## Setup

In [1]:
import bw2data
import bw2calc
import bw_eotw
from bw_eotw import set_config

bw2data.projects.set_current('eotw-getting-started')

# Start fresh
for name in list(bw2data.databases):
    del bw2data.databases[name]

print('Project:', bw2data.projects.current)

Project: eotw-getting-started


In [2]:
# ── biosphere ─────────────────────────────────────────────────────────────────
bio = bw2data.Database('biosphere')
bio.register()

co2 = bio.new_node(code='co2', name='Carbon dioxide', unit='kg', type='emission')
co2.save()
co2.new_edge(input=co2, type='production', amount=1.0).save()

# ── characterisation method ───────────────────────────────────────────────────
gwp = bw2data.Method(('GWP', 'simple'))
gwp.register(unit='kg CO2-eq')
gwp.write([(co2.key, 1.0)])

print(f'Biosphere: {len(bio)} flows  |  Method: {gwp.name}')

Biosphere: 1 flows  |  Method: ('GWP', 'simple')


In [3]:
# ── background electricity providers (standard SQLite backend) ────────────────
bg = bw2data.Database('background')
bg.register()

def make_provider(code, name, co2_per_kwh):
    node = bg.new_node(code=code, name=name, unit='kWh', type='process')
    node.save()
    node.new_edge(input=node, type='production', amount=1.0).save()
    node.new_edge(input=co2,  type='biosphere',  amount=co2_per_kwh).save()
    return node

coal  = make_provider('coal',  'Coal power',  0.82)
wind  = make_provider('wind',  'Wind power',  0.007)
solar = make_provider('solar', 'Solar power', 0.04)

print(coal, wind, solar)

'Coal power' (kWh, GLO, None) 'Wind power' (kWh, GLO, None) 'Solar power' (kWh, GLO, None)


In [4]:
# ── helpers used throughout the notebook ─────────────────────────────────────

def fresh_demo():
    """Delete and re-create the demo eotw database."""
    if 'demo' in bw2data.databases:
        del bw2data.databases['demo']
    db = bw2data.Database('demo', backend='eotw')
    db.register()
    return db

def make_widget(db):
    """Create a bare widget node with a production exchange."""
    w = db.new_node(code='widget', name='Widget production', unit='unit', type='process')
    w.save()
    w.new_edge(input=w, type='production', amount=1.0).save()
    return w

def run_lca(db, node):
    """Process the demo database and return the GWP score for 1 unit of node."""
    fu, data_objs, _ = bw2data.prepare_lca_inputs(
        {node: 1.0}, method=('GWP', 'simple'),
        remapping=False
    )
    lca = bw2calc.LCA(demand=fu, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    return lca

---
## 1 · Singlevalue interpreter

Behaves exactly like a plain bw2data exchange — one stored amount maps to one matrix cell.  
Widget uses **1.0 kWh of coal power**, so GWP = 1.0 × 0.82 = **0.82 kg CO₂-eq**.

In [5]:
db = fresh_demo()
widget = make_widget(db)

edge = widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='singlevalue',
    amount=1.0,
)
edge.save()

edge

input,"'Coal power' (kWh, GLO, None)"
output,"'Widget production' (unit, GLO, None)"
amount,1.0


In [6]:
lca = run_lca(db, widget)

print(lca.technosphere_matrix.toarray())

print(f'GWP = {lca.score:.4f} kg CO₂-eq  (expected {1.0 * 0.82:.4f})')

[[ 1.  0.  0. -1.]
 [ 0.  1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  1.]]
GWP = 0.8200 kg CO₂-eq  (expected 0.8200)


---
## 2 · Temporal interpreter

Stores a time-series in `temporal_values`.  At `process()` time, `config['year']` selects the active value.  
An optional per-edge `default_year` is used when the config year is absent or not found in the dict.

Here we model improving electricity efficiency (kWh consumed per widget decreases over time):

| year | kWh from coal | GWP |
|------|:---:|:---:|
| 2020 | 1.20 | 0.984 |
| 2025 | 1.00 | 0.820 |
| 2030 | 0.85 | 0.697 |

In [7]:
db = fresh_demo()
widget = make_widget(db)

edge = widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='temporal',
    temporal_values={2020: 1.20, 2025: 1.00, 2030: 0.85},
    default_year=2025,
)
edge.save()

edge

RichEdge(interpreter='temporal', input='Coal power' (kWh, GLO, None), output='Widget production' (unit, GLO, None), years=[2020, 2025, 2030], default_year=2025)

In [8]:
for year, expected in [(2020, 1.20 * 0.82), (2025, 1.00 * 0.82), (2030, 0.85 * 0.82)]:
    with db.set_config(config={"year": year}):
        lca = run_lca(db, widget)
        
        print(lca.technosphere_matrix.toarray())
        
        print(f'year={year}  GWP={lca.score:.4f}  (expected {expected:.4f})')

# default_year fallback when config year is missing
score = run_lca(db, widget)
print(f'\nno year in config → default_year=2025 → GWP={score:.4f}')

[[ 1.          0.          0.         -1.20000005]
 [ 0.          1.          0.          0.        ]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]
year=2020  GWP=0.9840  (expected 0.9840)


NonsquareTechnosphere: Technosphere matrix is not square: 4 activities (columns) and 5 products (rows). Use LeastSquaresLCA to solve this system, or fix the input data

In [10]:
%debug

    [... skipping 1 hidden frame(s)]
  /var/folders/rn/ht0vvs3s7mz2h9f_xjt9x4040000gn/T/ipykernel_31823/2627472725.py(3)<module>()
      1 for year, expected in [(2020, 1.20 * 0.82), (2025, 1.00 * 0.82), (2030, 0.85 * 0.82)]:
      2     with db.set_config(config={"year": year}):
----> 3         lca = run_lca(db, widget)
      4 
      5         print(lca.technosphere_matrix.toarray())
  /var/folders/rn/ht0vvs3s7mz2h9f_xjt9x4040000gn/T/ipykernel_31823/1823414357.py(25)run_lca()
     23     )
     24     lca = bw2calc.LCA(demand=fu, data_objs=data_objs)
---> 25     lca.lci()
     26     lca.lcia()
     27     return lca
  /Users/cmutel/Code/bw2/calc/src/bw2calc/lca_base.py(182)lci()
    180         """
    181         if not hasattr(self, "technosphere_matrix"):
--> 182             self.load_lci_data()
    183         if demand is not None:
    184             self.check_demand(demand)
> /Users/cmutel/Code/bw2/calc/src/bw2calc/lca_base.py(61)load_lci_data()
     59             and not n

ipdb>  self.technosphere_matrix.toarray()


array([[-1.,  0.,  0.,  0.],
       [ 1.,  0.,  0.,  0.],
       [ 0.,  1.,  0.,  0.],
       [ 0.,  0.,  1.,  0.],
       [ 0.,  0.,  0.,  1.]])


ipdb>  q


---
## 3 · Loss interpreter

Expands one edge into **two** matrix entries summed into the same cell:

```
main entry  = amount
loss entry  = amount × loss_factor
─────────────────────────────────
total       = amount × (1 + loss_factor)
```

With `amount=1.0` and `loss_factor=0.05`, the matrix cell becomes **1.05 kWh of coal**,  
so GWP = 1.05 × 0.82 = **0.861 kg CO₂-eq**.

In [21]:
db = fresh_demo()
widget = make_widget(db)

edge = widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='loss',
    amount=1.0,
    loss_factor=0.05,    # 5 % transmission loss
)
edge.save()

display(edge)

input,"('background', 'coal')"
output,"('demo', 'widget')"
amount,1.0
loss_factor,0.05


In [23]:
# Inspect the two resolved entries
score = run_lca(db, widget)
print(f'\nGWP = {score:.4f}  (expected {1.05 * 0.82:.4f})')


GWP = 0.8610  (expected 0.8610)


The value in the technosphere matrix is `1.0 - 0.05`:

In [ ]:
lca.technosphere_matrix[]

---
## 4 · Scenario interpreter

Stores one value per named scenario in `scenario_values`.  
`config['scenario']` **must** be present; a `KeyError` is raised if it's missing or unrecognised.

Here, different scenarios reflect different electricity efficiency assumptions (kWh/widget from coal):

| scenario | kWh | GWP |
|---|:---:|:---:|
| `coal_heavy` | 1.20 | 0.984 |
| `current`    | 1.00 | 0.820 |
| `efficient`  | 0.70 | 0.574 |

In [ ]:
db = fresh_demo()
widget = make_widget(db)

first = widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='scenario',
    scenario_values={
        'sunny':   0.6,
        'current': 1.00,
        'windy':   0.75,
    },
)
first.save()
first

In [ ]:
second = widget.new_edge(
    input=wind,
    type='technosphere',
    interpreter='scenario',
    scenario_values={
        'sunny':   0.2,
        'current': 0.0,
        'windy':   0.8,
    },
)
second.save()
second

In [ ]:
third = widget.new_edge(
    input=solar,
    type='technosphere',
    interpreter='scenario',
    scenario_values={
        'sunny':   0.8,
        'current': 0.0,
        'windy':   0.2,
    },
)
third.save()
third

In [30]:
for scenario in ['current', 'sunny', 'windy']:
    with set_config("demo", config={"scenario": scenario}):
        score = run_lca(db, widget)
        print(f"scenario={scenario:<12}  GWP={score:.4f}")

# Missing scenario key → KeyError
try:
    run_lca(db, widget)
except ValueError as exc:
    print(f'\nNo config → {exc}')

scenario=current       GWP=0.8200
scenario=sunny         GWP=0.5294
scenario=windy         GWP=0.6286


OutsideTechnosphere: Can't find key 306742319714381824 in product dictionary

---
## 5 · Provider-mix interpreter

Splits a single product demand across multiple provider nodes by fractional share,  
yielding one matrix entry per provider.  All shares must sum to 1.

Widget uses 1.0 kWh split: 60 % coal, 30 % wind, 10 % solar.

```
GWP = 0.60×0.820 + 0.30×0.007 + 0.10×0.040 = 0.492 + 0.0021 + 0.0040 ≈ 0.498 kg CO₂-eq
```

Compare to pure-coal (0.820): the mixed grid cuts GWP by ~39 %.

> **Note:** `mix` entries use integer node IDs (`.id`), not (database, code) tuples.

In [ ]:
db = fresh_demo()
widget = make_widget(db)

edge = widget.new_edge(
    input=coal,           # primary input for dependency tracking
    type='technosphere',
    interpreter='provider_mix',
    product_name='electricity',
    amount=1.0,
    mix=[
        {'input': coal.id,  'share': 0.60},
        {'input': wind.id,  'share': 0.30},
        {'input': solar.id, 'share': 0.10},
    ],
)
edge.save()

display(edge)

In [ ]:
# Inspect the three resolved entries
for entry in edge.resolve():
    print(f'row={entry.row}  col={entry.col}  amount={entry.amount}')

score = run_lca(db, widget)
expected = 0.60 * 0.82 + 0.30 * 0.007 + 0.10 * 0.04
print(f'\nGWP = {score:.4f}  (expected {expected:.4f})')
print(f'vs. pure coal: {1.0 * 0.82:.4f}  →  {(1 - score / 0.82) * 100:.1f} % reduction')

---
## 6 · Temporal-scenario interpreter

Combines strict scenario selection with year-keyed lookup — a grid of (scenario, year) values.  
Both `config['scenario']` and `config['year']` (or `default_year`) are required.

| | 2020 | 2030 |
|---|:---:|:---:|
| `current`     | 1.20 kWh → **0.984** | 0.90 kWh → **0.738** |
| `green_future`| 0.80 kWh → **0.656** | 0.30 kWh → **0.246** |

In [ ]:
db = fresh_demo()
widget = make_widget(db)

edge = widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='temporal_scenario',
    scenario_temporal_values={
        'current':      {2020: 1.20, 2030: 0.90},
        'green_future': {2020: 0.80, 2030: 0.30},
    },
    default_year=2020,
)
edge.save()

display(edge)

In [ ]:
cases = [
    ('current',      2020, 1.20 * 0.82),
    ('current',      2030, 0.90 * 0.82),
    ('green_future', 2020, 0.80 * 0.82),
    ('green_future', 2030, 0.30 * 0.82),
]

for scenario, year, expected in cases:
    score = run_lca(db, widget, config={'scenario': scenario, 'year': year})
    print(f"scenario={scenario:<14}  year={year}  GWP={score:.4f}  (expected {expected:.4f})")

# default_year fallback
score = run_lca(db, widget, config={'scenario': 'current'})
print(f'\nscenario=current, no year → default_year=2020 → GWP={score:.4f}')

---
## 7 · `set_config` — storing config in database metadata

`set_config(db_name, config)` writes the config into the database's metadata so that `db.process()` picks it up automatically — no need to pass `config=` on every call.  It also invalidates the cached datapackage when the config changes.

Used as a **context manager**, it restores the previous config on exit, making it safe to run comparative LCAs.

In [ ]:
db = fresh_demo()
widget = make_widget(db)
widget.new_edge(
    input=coal,
    type='technosphere',
    interpreter='temporal',
    amount=1.0,
    temporal_values={2020: 1.20, 2030: 0.85},
).save()

scores = {}
for year in [2020, 2030]:
    with set_config('demo', {'year': year}):
        db.process()          # reads config from metadata
        fu, do, rm = bw2data.prepare_lca_inputs({widget: 1.0}, method=('GWP', 'simple'))
        lca = bw2calc.LCA(demand=fu, data_objs=do, remapping_dicts=rm)
        lca.lci(); lca.lcia()
        scores[year] = lca.score
    # config is restored to whatever it was before the 'with' block

for year, score in scores.items():
    print(f'year={year}  GWP={score:.4f}')

print(f'\nConfig after context manager: {bw2data.databases["demo"].get("eotw_config")}')

---
## 8 · Validation at save time

`RichNode.new_edge(...).save()` runs `normalize_edge` and `validate_edge` before writing to the database, so structural errors are caught early.

In [ ]:
db = fresh_demo()
widget = make_widget(db)

# loss_factor outside [0, 1]
try:
    widget.new_edge(input=coal, type='technosphere',
                   interpreter='loss', amount=1.0, loss_factor=1.5).save()
except ValueError as exc:
    print(f'loss_factor=1.5 → {exc}')

# provider_mix shares not summing to 1
try:
    widget.new_edge(
        input=coal, type='technosphere',
        interpreter='provider_mix', amount=1.0,
        product_name='electricity',
        mix=[
            {'input': coal.id, 'share': 0.4},
            {'input': wind.id, 'share': 0.4},   # total = 0.8 ≠ 1
        ],
    ).save()
except ValueError as exc:
    print(f'bad shares → {exc}')

# temporal_scenario with default_year missing from one scenario
try:
    widget.new_edge(
        input=coal, type='technosphere',
        interpreter='temporal_scenario',
        scenario_temporal_values={
            'a': {2020: 1.0},
            'b': {2030: 0.7},   # 2020 absent here
        },
        default_year=2020,
    ).save()
except ValueError as exc:
    print(f'default_year mismatch → {exc}')

---
## Cleanup

In [ ]:
for name in list(bw2data.databases):
    del bw2data.databases[name]

print('All databases removed.')